In [2]:
import pandas as pd
import numpy as np

In [3]:
df1=pd.read_csv(r"D:\code_of_digital_impersonation\unique_professional_identities_500_repaired.csv")
df2=pd.read_csv(r"D:\code_of_digital_impersonation\dataset2_500_final_v7.csv")
df3=pd.read_csv(r"D:\code_of_digital_impersonation\dataset3_500_final_v4.csv")

# final code 

In [17]:
# ============================================================
# CYBERGUARD — COMPLETE END-TO-END PIPELINE
# Dataset 1 → Model 1 → Model 2 → Model 3 → Risk Engine
# ============================================================

import joblib
import pandas as pd
import re

# ============================================================
# 1. LOAD SAVED MODELS
# ============================================================

model_1 = joblib.load(
    "cyberguard_model_1_impersonation.pkl"
)

tfidf_1 = joblib.load(
    "cyberguard_model_1_tfidf.pkl"
)

model_2 = joblib.load(
    "cyberguard_model_2_attack_type.pkl"
)

tfidf_2 = joblib.load(
    "cyberguard_model_2_tfidf.pkl"
)

model_3 = joblib.load(
    "cyberguard_model_3_behavioral.pkl"
)

# Dataset 1 must already be loaded as df1
if "df1" not in globals():
    raise RuntimeError(
        "Dataset 1 is not loaded as df1."
    )

# ============================================================
# 2. EMAIL INPUT
# ============================================================
# Change ONLY this section when testing another email.

sender_name = "Rajesh Kumar"
sender_email = "raj.kumar@gmail.com"

receiver_name = "Finance Officer"
receiver_email = "finance@example.org"

reply_to = ""

date_time = "2026-09-23 14:30"

subject = "Urgent: Verify your account immediately"

message = """
Dear Finance Officer,

Your account requires immediate verification. Please confirm your
login credentials and verification code as soon as possible to avoid
suspension of your account.

Regards,
Rajesh Kumar
"""

# ============================================================
# 3. BASIC EXTRACTION
# ============================================================

text_input = subject + " " + message

sender_domain = (
    sender_email.split("@")[-1].lower()
    if "@" in sender_email
    else ""
)

receiver_domain = (
    receiver_email.split("@")[-1].lower()
    if "@" in receiver_email
    else ""
)

reply_to_domain = (
    reply_to.split("@")[-1].lower()
    if reply_to and "@" in reply_to
    else ""
)

message_length = len(message.strip())

dt = pd.to_datetime(date_time)

current_hour = dt.hour
current_day = dt.day_name()

if 5 <= current_hour < 12:
    current_period = "Morning"
elif 12 <= current_hour < 17:
    current_period = "Afternoon"
elif 17 <= current_hour < 21:
    current_period = "Evening"
else:
    current_period = "Night"

# ============================================================
# 4. DATASET 1 — IDENTITY ANALYSIS
# ============================================================

identity_matches = df1[
    df1["official_email"]
    .astype(str)
    .str.lower()
    == sender_email.lower()
]

identity_found = len(identity_matches) > 0

if identity_found:

    identity = identity_matches.iloc[0]

    identity_id = identity["identity_id"]
    identity_name = identity["name"]
    identity_role = identity["role"]
    identity_organization = identity["organization"]
    official_domain = str(
        identity["official_domain"]
    ).lower()

else:

    identity = None

    identity_id = None
    identity_name = None
    identity_role = None
    identity_organization = None
    official_domain = None

# ============================================================
# 5. MODEL 1 — IMPERSONATION DETECTION
# ============================================================

X_m1 = tfidf_1.transform(
    [text_input]
)

m1_probabilities = model_1.predict_proba(
    X_m1
)[0]

legitimate_probability = float(
    m1_probabilities[0]
)

impersonation_probability = float(
    m1_probabilities[1]
)

if impersonation_probability >= 0.5:
    model1_prediction = "IMPERSONATION"
else:
    model1_prediction = "LEGITIMATE"

# ============================================================
# 6. MODEL 2 — ATTACK TYPE CLASSIFICATION
# ============================================================

X_m2 = tfidf_2.transform(
    [text_input]
)

m2_probabilities = model_2.predict_proba(
    X_m2
)[0]

m2_index = m2_probabilities.argmax()

model2_prediction = str(
    model_2.classes_[m2_index]
)

model2_confidence = float(
    m2_probabilities[m2_index]
)

# Top 3 attack types
top3_indices = m2_probabilities.argsort()[-3:][::-1]

top3_attacks = [
    (
        str(model_2.classes_[i]),
        float(m2_probabilities[i])
    )
    for i in top3_indices
]

# ============================================================
# 7. MODEL 3 — BEHAVIORAL ANALYSIS
# ============================================================

profiles = model_3["dataset3_profiles"]
weights = model_3["weights"]

model3_status = "UNKNOWN"
model3_score = None

behavior_details = {
    "time_match": None,
    "day_match": None,
    "domain_match": None,
    "length_match": None,
}

if identity_found:

    profile_matches = profiles[
        profiles["identity_id"]
        == identity_id
    ]

    if len(profile_matches) > 0:

        profile = profile_matches.iloc[0]

        # --------------------------------------------
        # Time
        # --------------------------------------------

        time_match = (
            current_period.lower()
            in str(profile["usual_hours"]).lower()
        )

        # --------------------------------------------
        # Day
        # --------------------------------------------

        usual_days = str(
            profile["usual_days"]
        ).lower()

        if "all days" in usual_days:

            day_match = True

        elif "weekdays" in usual_days:

            day_match = current_day not in [
                "Saturday",
                "Sunday"
            ]

        else:

            day_match = (
                current_day.lower()
                in usual_days
            )

        # --------------------------------------------
        # Sender domain
        # --------------------------------------------

        behavior_domain = str(
            profile["usual_sender_domain"]
        ).lower()

        domain_match = (
            sender_domain
            == behavior_domain
        )

        # --------------------------------------------
        # Message length
        # --------------------------------------------

        avg_length = float(
            profile["avg_message_length"]
        )

        if avg_length > 0:

            length_ratio = (
                message_length
                / avg_length
            )

            length_match = (
                0.5
                <= length_ratio
                <= 1.5
            )

        else:

            length_ratio = None
            length_match = False

        # --------------------------------------------
        # Weighted behavioral score
        # --------------------------------------------

        model3_score = (

            weights["time_of_day"]
            * (0 if time_match else 1)

            +

            weights["day_of_week"]
            * (0 if day_match else 1)

            +

            weights["sender_domain"]
            * (0 if domain_match else 1)

            +

            weights["message_length"]
            * (0 if length_match else 1)
        )

        if model3_score >= 0.5:
            model3_status = "ANOMALOUS"
        else:
            model3_status = "NORMAL"

        behavior_details = {
            "time_match": time_match,
            "day_match": day_match,
            "domain_match": domain_match,
            "length_match": length_match,
            "length_ratio": (
                length_ratio
                if avg_length > 0
                else None
            )
        }

# ============================================================
# 8. TECHNICAL / CONTENT SIGNALS
# ============================================================

public_email_domains = {
    "gmail.com",
    "yahoo.com",
    "outlook.com",
    "hotmail.com",
    "proton.me",
    "protonmail.com",
    "icloud.com"
}

public_email = (
    sender_domain
    in public_email_domains
)

# Sender domain vs Dataset 1
if identity_found:

    sender_domain_matches_identity = (
        sender_domain
        == official_domain
    )

else:

    sender_domain_matches_identity = False

# Reply-To
if reply_to:

    reply_to_matches_sender = (
        reply_to_domain
        == sender_domain
    )

else:

    # No Reply-To is not a mismatch
    reply_to_matches_sender = True

# ============================================================
# 9. CONTENT SIGNAL DETECTION
# ============================================================

lower_text = text_input.lower()

urgency_words = [
    "urgent",
    "immediately",
    "as soon as possible",
    "right away",
    "within",
    "suspended",
    "suspension",
    "today",
    "now"
]

sensitive_words = [
    "password",
    "login credentials",
    "credential",
    "verification code",
    "otp",
    "pin",
    "account details",
    "bank details",
    "financial information"
]

urgency_detected = any(
    word in lower_text
    for word in urgency_words
)

sensitive_request_detected = any(
    word in lower_text
    for word in sensitive_words
)

# ============================================================
# 10. LINK / ATTACHMENT SIGNALS
# ============================================================

url_pattern = r"https?://[^\s]+|www\.[^\s]+"

links = re.findall(
    url_pattern,
    text_input,
    flags=re.IGNORECASE
)

has_link = len(links) > 0

attachment_extensions = [
    ".pdf",
    ".doc",
    ".docx",
    ".xls",
    ".xlsx",
    ".zip",
    ".rar",
    ".exe"
]

has_attachment = any(
    ext in lower_text
    for ext in attachment_extensions
)

# ============================================================
# 11. RISK / EVIDENCE FUSION
# ============================================================

# Initial engineering weights.
# These are NOT calibrated probabilities.

MAX_MODEL1 = 25
UNKNOWN_IDENTITY = 10
DOMAIN_MISMATCH = 25
REPLY_TO_MISMATCH = 15
PUBLIC_EMAIL = 10
URGENCY = 15
SENSITIVE_REQUEST = 20
MAX_MODEL3 = 15

risk_contributions = {}

# Model 1
risk_contributions[
    "Model 1 impersonation"
] = (
    impersonation_probability
    * MAX_MODEL1
)

# Unknown identity
risk_contributions[
    "Unknown identity"
] = (
    UNKNOWN_IDENTITY
    if not identity_found
    else 0
)

# Domain mismatch
risk_contributions[
    "Domain mismatch"
] = (
    DOMAIN_MISMATCH
    if (
        identity_found
        and not sender_domain_matches_identity
    )
    else 0
)

# Reply-To mismatch
risk_contributions[
    "Reply-To mismatch"
] = (
    REPLY_TO_MISMATCH
    if not reply_to_matches_sender
    else 0
)

# Public email
risk_contributions[
    "Public email domain"
] = (
    PUBLIC_EMAIL
    if public_email
    else 0
)

# Urgency
risk_contributions[
    "Urgency"
] = (
    URGENCY
    if urgency_detected
    else 0
)

# Sensitive request
risk_contributions[
    "Sensitive request"
] = (
    SENSITIVE_REQUEST
    if sensitive_request_detected
    else 0
)

# Model 3
if model3_score is not None:

    risk_contributions[
        "Model 3 behavior"
    ] = (
        model3_score
        * MAX_MODEL3
    )

else:

    risk_contributions[
        "Model 3 behavior"
    ] = 0

# ============================================================
# 12. FINAL RISK SCORE
# ============================================================

risk_score = sum(
    risk_contributions.values()
)

risk_score = min(
    100,
    risk_score
)

# ============================================================
# 13. RISK BAND
# ============================================================

if risk_score <= 20:

    risk_band = "SAFE"

elif risk_score <= 40:

    risk_band = "LOW"

elif risk_score <= 60:

    risk_band = "MEDIUM"

elif risk_score <= 80:

    risk_band = "HIGH"

else:

    risk_band = "CRITICAL"

# ============================================================
# 14. EXPLANATION GENERATION
# ============================================================

reasons = []

if impersonation_probability >= 0.5:

    reasons.append(
        "Model 1 detected impersonation indicators."
    )

if not identity_found:

    reasons.append(
        "Sender identity was not found in Dataset 1."
    )

if public_email:

    reasons.append(
        "Sender uses a public/free email domain."
    )

if urgency_detected:

    reasons.append(
        "Message contains urgency indicators."
    )

if sensitive_request_detected:

    reasons.append(
        "Message requests sensitive authentication information."
    )

if (
    identity_found
    and not sender_domain_matches_identity
):

    reasons.append(
        "Sender domain does not match the known identity."
    )

if not reply_to_matches_sender:

    reasons.append(
        "Reply-To domain does not match the sender domain."
    )

if model3_status == "ANOMALOUS":

    reasons.append(
        "Sender behavior is anomalous for the identified identity."
    )

elif model3_status == "NORMAL":

    reasons.append(
        "Sender behavior is consistent with the learned profile."
    )

else:

    reasons.append(
        "Behavioral analysis is unavailable because "
        "the sender identity is unknown."
    )

# ============================================================
# 15. FINAL CYBERGUARD REPORT
# ============================================================

print("\n")
print("=" * 90)
print("                 CYBERGUARD — FINAL ANALYSIS")
print("=" * 90)

print("\n[1] EMAIL")
print("-" * 90)
print("Sender           :", sender_name)
print("Sender email     :", sender_email)
print("Receiver         :", receiver_name)
print("Receiver email   :", receiver_email)
print("Subject          :", subject)
print("Date/time        :", date_time)

print("\n[2] IDENTITY ANALYSIS — DATASET 1")
print("-" * 90)
print("Identity found   :", identity_found)

if identity_found:

    print("Identity ID      :", identity_id)
    print("Identity name    :", identity_name)
    print("Role             :", identity_role)
    print("Organization     :", identity_organization)
    print("Official domain  :", official_domain)

else:

    print("Identity ID      : None")
    print("Status           : UNKNOWN SENDER")

print("\n[3] MODEL 1 — IMPERSONATION DETECTION")
print("-" * 90)
print("Prediction       :", model1_prediction)
print(
    "Legitimate       :",
    f"{legitimate_probability:.4f}"
)
print(
    "Impersonation    :",
    f"{impersonation_probability:.4f}"
)

print("\n[4] MODEL 2 — ATTACK TYPE")
print("-" * 90)
print("Attack type      :", model2_prediction)
print(
    "Confidence       :",
    f"{model2_confidence:.4f}"
)

print("\nTop 3 attack types:")

for attack, probability in top3_attacks:

    print(
        f"  {attack:<25} "
        f"{probability:.4f}"
    )

print("\n[5] MODEL 3 — BEHAVIORAL ANALYSIS")
print("-" * 90)
print("Status           :", model3_status)

if model3_score is not None:

    print(
        "Behavior score   :",
        f"{model3_score:.4f}"
    )

    print(
        "Time match       :",
        behavior_details["time_match"]
    )

    print(
        "Day match        :",
        behavior_details["day_match"]
    )

    print(
        "Domain match     :",
        behavior_details["domain_match"]
    )

    print(
        "Length match     :",
        behavior_details["length_match"]
    )

else:

    print("Behavior score   : N/A")

print("\n[6] TECHNICAL / CONTENT ANALYSIS")
print("-" * 90)
print("Sender domain    :", sender_domain)
print("Receiver domain  :", receiver_domain)
print(
    "Public email     :",
    public_email
)
print(
    "Domain mismatch  :",
    (
        identity_found
        and not sender_domain_matches_identity
    )
)
print(
    "Reply-To mismatch:",
    not reply_to_matches_sender
)
print(
    "Urgency          :",
    urgency_detected
)
print(
    "Sensitive request:",
    sensitive_request_detected
)
print(
    "Has link         :",
    has_link
)
print(
    "Has attachment   :",
    has_attachment
)

print("\n[7] RISK / EVIDENCE FUSION")
print("-" * 90)

for name, value in risk_contributions.items():

    print(
        f"{name:<30}: +{value:.2f}"
    )

print("-" * 90)

print(
    "FINAL RISK SCORE :",
    f"{risk_score:.2f} / 100"
)

print(
    "RISK LEVEL       :",
    risk_band
)

print("\n[8] EXPLANATION")
print("-" * 90)

for i, reason in enumerate(
    reasons,
    start=1
):

    print(
        f"{i}. {reason}"
    )

print("\n" + "=" * 90)
print("                 CYBERGUARD ANALYSIS COMPLETE")
print("=" * 90)



                 CYBERGUARD — FINAL ANALYSIS

[1] EMAIL
------------------------------------------------------------------------------------------
Sender           : Rajesh Kumar
Sender email     : raj.kumar@gmail.com
Receiver         : Finance Officer
Receiver email   : finance@example.org
Subject          : Urgent: Verify your account immediately
Date/time        : 2026-09-23 14:30

[2] IDENTITY ANALYSIS — DATASET 1
------------------------------------------------------------------------------------------
Identity found   : False
Identity ID      : None
Status           : UNKNOWN SENDER

[3] MODEL 1 — IMPERSONATION DETECTION
------------------------------------------------------------------------------------------
Prediction       : IMPERSONATION
Legitimate       : 0.3049
Impersonation    : 0.6951

[4] MODEL 2 — ATTACK TYPE
------------------------------------------------------------------------------------------
Attack type      : CREDENTIAL_REQUEST
Confidence       : 0.4465

Top 

In [19]:
# ============================================================
# CYBERGUARD — COMPLETE END-TO-END PIPELINE
# ============================================================
#
# Dataset 1 → Identity Analysis
#          ↓
# Model 1 → Impersonation Detection
#          ↓
# Model 2 → Attack Type Classification
#          ↓
# Model 3 → Identity-Specific Behavioral Analysis
#          ↓
# Technical / Content Analysis
#          ↓
# Risk & Evidence Fusion
#          ↓
# Final Risk Score + Explanation
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import joblib
import pandas as pd
import re


# ============================================================
# 1. LOAD SAVED MODELS
# ============================================================

model_1 = joblib.load(
    "cyberguard_model_1_impersonation.pkl"
)

tfidf_1 = joblib.load(
    "cyberguard_model_1_tfidf.pkl"
)

model_2 = joblib.load(
    "cyberguard_model_2_attack_type.pkl"
)

tfidf_2 = joblib.load(
    "cyberguard_model_2_tfidf.pkl"
)

model_3 = joblib.load(
    "cyberguard_model_3_behavioral.pkl"
)


# ============================================================
# 2. CHECK DATASET 1
# ============================================================

if "df1" not in globals():

    raise RuntimeError(
        "Dataset 1 is not loaded as df1."
    )


required_df1_columns = [
    "identity_id",
    "name",
    "role",
    "organization",
    "official_email",
    "official_domain"
]

missing_df1_columns = [
    col
    for col in required_df1_columns
    if col not in df1.columns
]

if missing_df1_columns:

    raise RuntimeError(
        f"Dataset 1 is missing columns: "
        f"{missing_df1_columns}"
    )


# ============================================================
# 3. EMAIL INPUT
# ============================================================
#
# Change ONLY this section when testing another email.
#
# ============================================================

sender_name = "Rajesh Kumar"

sender_email = "raj.kumar@gmail.com"

receiver_name = "Finance Officer"

receiver_email = "finance@example.org"

reply_to = ""

date_time = "2026-09-23 14:30"

subject = "Urgent: Verify your account immediately"

message = """
Dear Finance Officer,

Your account requires immediate verification. Please confirm your
login credentials and verification code as soon as possible to avoid
suspension of your account.

Regards,
Rajesh Kumar
"""


# ============================================================
# 4. BASIC INPUT VALIDATION
# ============================================================

sender_email = str(sender_email).strip()

receiver_email = str(receiver_email).strip()

reply_to = str(reply_to).strip()

sender_name = str(sender_name).strip()

receiver_name = str(receiver_name).strip()

subject = str(subject).strip()

message = str(message).strip()


if not sender_email:

    raise ValueError(
        "Sender email cannot be empty."
    )


if not subject:

    raise ValueError(
        "Subject cannot be empty."
    )


if not message:

    raise ValueError(
        "Message cannot be empty."
    )


# ============================================================
# 5. BASIC EXTRACTION
# ============================================================

text_input = (
    subject
    + " "
    + message
)


# Sender domain

sender_domain = (
    sender_email.split("@")[-1].lower()
    if "@" in sender_email
    else ""
)


# Receiver domain

receiver_domain = (
    receiver_email.split("@")[-1].lower()
    if "@" in receiver_email
    else ""
)


# Reply-To domain

reply_to_domain = (
    reply_to.split("@")[-1].lower()
    if reply_to and "@" in reply_to
    else ""
)


# Message length

message_length = len(
    message.strip()
)


# Date / time

dt = pd.to_datetime(
    date_time
)


current_hour = dt.hour

current_day = dt.day_name()


# Time period

if 5 <= current_hour < 12:

    current_period = "Morning"

elif 12 <= current_hour < 17:

    current_period = "Afternoon"

elif 17 <= current_hour < 21:

    current_period = "Evening"

else:

    current_period = "Night"


# ============================================================
# 6. DATASET 1 — ROBUST IDENTITY ANALYSIS
# ============================================================

def normalize_name(name):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(name).lower()
    )


# Find every identity using this official email

identity_matches = df1[
    df1["official_email"]
    .astype(str)
    .str.strip()
    .str.lower()
    ==
    sender_email.lower()
].copy()


# Default identity values

identity_found = False

identity_ambiguous = False

identity = None

identity_id = None

identity_name = None

identity_role = None

identity_organization = None

official_domain = None


# ------------------------------------------------------------
# CASE 1 — No identity found
# ------------------------------------------------------------

if len(identity_matches) == 0:

    identity_found = False

    identity_ambiguous = False


# ------------------------------------------------------------
# CASE 2 — Exactly one identity found
# ------------------------------------------------------------

elif len(identity_matches) == 1:

    identity_found = True

    identity_ambiguous = False

    identity = identity_matches.iloc[0]

    identity_id = identity["identity_id"]

    identity_name = identity["name"]

    identity_role = identity["role"]

    identity_organization = identity["organization"]

    official_domain = str(
        identity["official_domain"]
    ).strip().lower()


# ------------------------------------------------------------
# CASE 3 — Shared email
# ------------------------------------------------------------

else:

    normalized_sender_name = normalize_name(
        sender_name
    )

    identity_matches["normalized_name"] = (
        identity_matches["name"]
        .apply(normalize_name)
    )

    name_matches = identity_matches[
        identity_matches["normalized_name"]
        ==
        normalized_sender_name
    ].copy()


    # ----------------------------------------
    # Unique email + name match
    # ----------------------------------------

    if len(name_matches) == 1:

        identity_found = True

        identity_ambiguous = False

        identity = name_matches.iloc[0]

        identity_id = identity["identity_id"]

        identity_name = identity["name"]

        identity_role = identity["role"]

        identity_organization = identity["organization"]

        official_domain = str(
            identity["official_domain"]
        ).strip().lower()


    # ----------------------------------------
    # Still ambiguous
    # ----------------------------------------

    else:

        identity_found = False

        identity_ambiguous = True

        identity = None

        identity_id = None

        identity_name = None

        identity_role = None

        identity_organization = None

        official_domain = None


# ============================================================
# 7. MODEL 1 — IMPERSONATION DETECTION
# ============================================================

X_m1 = tfidf_1.transform(
    [text_input]
)


m1_probabilities = model_1.predict_proba(
    X_m1
)[0]


# Model 1 classes:
# 0 = LEGITIMATE
# 1 = IMPERSONATION

class_to_probability_m1 = {
    int(cls): float(prob)
    for cls, prob
    in zip(
        model_1.classes_,
        m1_probabilities
    )
}


legitimate_probability = (
    class_to_probability_m1.get(
        0,
        0.0
    )
)


impersonation_probability = (
    class_to_probability_m1.get(
        1,
        0.0
    )
)


if impersonation_probability >= 0.5:

    model1_prediction = "IMPERSONATION"

else:

    model1_prediction = "LEGITIMATE"


# ============================================================
# 8. MODEL 2 — ATTACK TYPE CLASSIFICATION
# ============================================================
#
# IMPORTANT:
# Model 2 was trained ONLY on IMPERSONATION records.
#
# Therefore Model 2 does NOT determine whether an email is
# legitimate or malicious.
#
# It identifies the likely attack type.
#
# ============================================================

X_m2 = tfidf_2.transform(
    [text_input]
)


m2_probabilities = model_2.predict_proba(
    X_m2
)[0]


m2_index = m2_probabilities.argmax()


model2_prediction = str(
    model_2.classes_[m2_index]
)


model2_confidence = float(
    m2_probabilities[m2_index]
)


# Top 3 attack types

top3_indices = (
    m2_probabilities
    .argsort()[-3:][::-1]
)


top3_attacks = [

    (
        str(model_2.classes_[i]),
        float(m2_probabilities[i])
    )

    for i in top3_indices
]


# ============================================================
# 9. MODEL 3 — IDENTITY-SPECIFIC BEHAVIORAL ANALYSIS
# ============================================================

profiles = model_3[
    "dataset3_profiles"
]

weights = model_3[
    "weights"
]


model3_status = "UNKNOWN"

model3_score = None


behavior_details = {

    "time_match": None,

    "day_match": None,

    "domain_match": None,

    "length_match": None,

    "length_ratio": None

}


# ------------------------------------------------------------
# Behavioral analysis only possible when identity is known
# ------------------------------------------------------------

if identity_found:

    profile_matches = profiles[
        profiles["identity_id"]
        ==
        identity_id
    ]


    if len(profile_matches) > 0:

        profile = profile_matches.iloc[0]


        # ----------------------------------------------------
        # TIME
        # ----------------------------------------------------

        time_match = (

            current_period.lower()

            in

            str(
                profile["usual_hours"]
            ).lower()

        )


        # ----------------------------------------------------
        # DAY
        # ----------------------------------------------------

        usual_days = str(
            profile["usual_days"]
        ).lower()


        if "all days" in usual_days:

            day_match = True


        elif "weekdays" in usual_days:

            day_match = (
                current_day
                not in
                [
                    "Saturday",
                    "Sunday"
                ]
            )


        else:

            day_match = (
                current_day.lower()
                in
                usual_days
            )


        # ----------------------------------------------------
        # SENDER DOMAIN
        # ----------------------------------------------------

        behavior_domain = str(
            profile["usual_sender_domain"]
        ).strip().lower()


        domain_match = (
            sender_domain
            ==
            behavior_domain
        )


        # ----------------------------------------------------
        # MESSAGE LENGTH
        # ----------------------------------------------------

        avg_length = float(
            profile["avg_message_length"]
        )


        if avg_length > 0:

            length_ratio = (
                message_length
                /
                avg_length
            )


            length_match = (
                0.5
                <=
                length_ratio
                <=
                1.5
            )


        else:

            length_ratio = None

            length_match = False


        # ----------------------------------------------------
        # BEHAVIORAL SCORE
        # ----------------------------------------------------
        #
        # 0 = completely consistent
        # 1 = all four behavioral signals anomalous
        #
        # ----------------------------------------------------

        model3_score = (

            weights["time_of_day"]
            *
            (0 if time_match else 1)

            +

            weights["day_of_week"]
            *
            (0 if day_match else 1)

            +

            weights["sender_domain"]
            *
            (0 if domain_match else 1)

            +

            weights["message_length"]
            *
            (0 if length_match else 1)

        )


        if model3_score >= 0.5:

            model3_status = "ANOMALOUS"

        else:

            model3_status = "NORMAL"


        behavior_details = {

            "time_match":
                time_match,

            "day_match":
                day_match,

            "domain_match":
                domain_match,

            "length_match":
                length_match,

            "length_ratio":
                length_ratio

        }


# ============================================================
# 10. TECHNICAL SIGNALS
# ============================================================

public_email_domains = {

    "gmail.com",

    "yahoo.com",

    "outlook.com",

    "hotmail.com",

    "proton.me",

    "protonmail.com",

    "icloud.com"

}


public_email = (
    sender_domain
    in public_email_domains
)


# ------------------------------------------------------------
# Sender domain vs known identity
# ------------------------------------------------------------

if identity_found:

    sender_domain_matches_identity = (

        sender_domain
        ==
        official_domain

    )

else:

    sender_domain_matches_identity = None


# ------------------------------------------------------------
# Reply-To
# ------------------------------------------------------------

if reply_to:

    reply_to_matches_sender = (

        reply_to_domain
        ==
        sender_domain

    )

else:

    # No Reply-To is NOT a mismatch.

    reply_to_matches_sender = True


# ============================================================
# 11. CONTENT SIGNAL DETECTION
# ============================================================

lower_text = text_input.lower()


# ------------------------------------------------------------
# Urgency
# ------------------------------------------------------------

urgency_words = [

    "urgent",

    "immediately",

    "as soon as possible",

    "right away",

    "within",

    "suspended",

    "suspension",

    "today",

    "now"

]


urgency_detected = any(

    word in lower_text

    for word in urgency_words

)


# ------------------------------------------------------------
# Sensitive information request
# ------------------------------------------------------------

sensitive_words = [

    "password",

    "login credentials",

    "credential",

    "verification code",

    "otp",

    "pin",

    "account details",

    "bank details",

    "financial information"

]


sensitive_request_detected = any(

    word in lower_text

    for word in sensitive_words

)


# ============================================================
# 12. LINK DETECTION
# ============================================================

url_pattern = (
    r"https?://[^\s]+"
    r"|www\.[^\s]+"
)


links = re.findall(

    url_pattern,

    text_input,

    flags=re.IGNORECASE

)


has_link = (
    len(links) > 0
)


# ============================================================
# 13. ATTACHMENT DETECTION
# ============================================================

attachment_extensions = [

    ".pdf",

    ".doc",

    ".docx",

    ".xls",

    ".xlsx",

    ".zip",

    ".rar",

    ".exe"

]


has_attachment = any(

    ext in lower_text

    for ext in attachment_extensions

)


# ============================================================
# 14. RISK / EVIDENCE FUSION
# ============================================================
#
# IMPORTANT:
# This is an engineering risk score.
# It is NOT a calibrated probability.
#
# ============================================================

MAX_MODEL1 = 25

UNKNOWN_IDENTITY = 10

DOMAIN_MISMATCH = 25

REPLY_TO_MISMATCH = 15

PUBLIC_EMAIL = 10

URGENCY = 15

SENSITIVE_REQUEST = 20

MAX_MODEL3 = 15


risk_contributions = {}


# ------------------------------------------------------------
# Model 1
# ------------------------------------------------------------

risk_contributions[
    "Model 1 impersonation"
] = (

    impersonation_probability
    *
    MAX_MODEL1

)


# ------------------------------------------------------------
# Unknown / ambiguous identity
# ------------------------------------------------------------
#
# Both UNKNOWN and AMBIGUOUS mean we cannot confidently
# establish the sender identity.
#
# ------------------------------------------------------------

risk_contributions[
    "Unknown / ambiguous identity"
] = (

    UNKNOWN_IDENTITY

    if (
        not identity_found
    )

    else 0

)


# ------------------------------------------------------------
# Domain mismatch
# ------------------------------------------------------------
#
# Only evaluate this when identity is actually known.
#
# Unknown sender ≠ domain mismatch.
#
# ------------------------------------------------------------

risk_contributions[
    "Domain mismatch"
] = (

    DOMAIN_MISMATCH

    if (

        identity_found

        and

        not sender_domain_matches_identity

    )

    else 0

)


# ------------------------------------------------------------
# Reply-To mismatch
# ------------------------------------------------------------

risk_contributions[
    "Reply-To mismatch"
] = (

    REPLY_TO_MISMATCH

    if not reply_to_matches_sender

    else 0

)


# ------------------------------------------------------------
# Public email domain
# ------------------------------------------------------------

risk_contributions[
    "Public email domain"
] = (

    PUBLIC_EMAIL

    if public_email

    else 0

)


# ------------------------------------------------------------
# Urgency
# ------------------------------------------------------------

risk_contributions[
    "Urgency"
] = (

    URGENCY

    if urgency_detected

    else 0

)


# ------------------------------------------------------------
# Sensitive request
# ------------------------------------------------------------

risk_contributions[
    "Sensitive request"
] = (

    SENSITIVE_REQUEST

    if sensitive_request_detected

    else 0

)


# ------------------------------------------------------------
# Model 3
# ------------------------------------------------------------

if model3_score is not None:

    risk_contributions[
        "Model 3 behavior"
    ] = (

        model3_score
        *
        MAX_MODEL3

    )

else:

    risk_contributions[
        "Model 3 behavior"
    ] = 0


# ============================================================
# 15. FINAL RISK SCORE
# ============================================================

risk_score = sum(
    risk_contributions.values()
)


risk_score = min(
    100,
    risk_score
)


# ============================================================
# 16. RISK BAND
# ============================================================

if risk_score <= 20:

    risk_band = "SAFE"

elif risk_score <= 40:

    risk_band = "LOW"

elif risk_score <= 60:

    risk_band = "MEDIUM"

elif risk_score <= 80:

    risk_band = "HIGH"

else:

    risk_band = "CRITICAL"


# ============================================================
# 17. EXPLANATION GENERATION
# ============================================================

reasons = []


# Model 1

if impersonation_probability >= 0.5:

    reasons.append(
        "Model 1 detected impersonation indicators."
    )


# Identity

if identity_ambiguous:

    reasons.append(
        "Sender email is shared by multiple Dataset 1 "
        "identities and could not be uniquely resolved."
    )

elif not identity_found:

    reasons.append(
        "Sender identity was not found in Dataset 1."
    )


# Public email

if public_email:

    reasons.append(
        "Sender uses a public/free email domain."
    )


# Domain mismatch

if (

    identity_found

    and

    not sender_domain_matches_identity

):

    reasons.append(
        "Sender domain does not match the known identity."
    )


# Reply-To

if not reply_to_matches_sender:

    reasons.append(
        "Reply-To domain does not match the sender domain."
    )


# Urgency

if urgency_detected:

    reasons.append(
        "Message contains urgency indicators."
    )


# Sensitive request

if sensitive_request_detected:

    reasons.append(
        "Message requests sensitive authentication "
        "or account information."
    )


# Link

if has_link:

    reasons.append(
        f"Message contains {len(links)} link(s)."
    )


# Attachment

if has_attachment:

    reasons.append(
        "Message appears to reference an attachment."
    )


# Model 3

if model3_status == "ANOMALOUS":

    reasons.append(
        "Sender behavior is anomalous for the "
        "identified identity."
    )

elif model3_status == "NORMAL":

    reasons.append(
        "Sender behavior is consistent with the "
        "learned identity profile."
    )

else:

    reasons.append(
        "Behavioral analysis is unavailable because "
        "the sender identity is unknown or ambiguous."
    )


# ============================================================
# 18. FINAL CYBERGUARD REPORT
# ============================================================

print("\n")

print("=" * 90)

print(
    "                 CYBERGUARD — FINAL ANALYSIS"
)

print("=" * 90)


# ------------------------------------------------------------
# EMAIL
# ------------------------------------------------------------

print("\n[1] EMAIL")

print("-" * 90)

print(
    "Sender           :",
    sender_name
)

print(
    "Sender email     :",
    sender_email
)

print(
    "Receiver         :",
    receiver_name
)

print(
    "Receiver email   :",
    receiver_email
)

print(
    "Subject          :",
    subject
)

print(
    "Date/time        :",
    date_time
)


# ------------------------------------------------------------
# DATASET 1
# ------------------------------------------------------------

print(
    "\n[2] IDENTITY ANALYSIS — DATASET 1"
)

print("-" * 90)

print(
    "Identity found   :",
    identity_found
)

print(
    "Identity ambiguous:",
    identity_ambiguous
)


if identity_found:

    print(
        "Identity ID      :",
        identity_id
    )

    print(
        "Identity name    :",
        identity_name
    )

    print(
        "Role             :",
        identity_role
    )

    print(
        "Organization     :",
        identity_organization
    )

    print(
        "Official domain  :",
        official_domain
    )


elif identity_ambiguous:

    print(
        "Status           : AMBIGUOUS SHARED EMAIL"
    )

    print(
        "Possible matches :",
        len(identity_matches)
    )


else:

    print(
        "Identity ID      : None"
    )

    print(
        "Status           : UNKNOWN SENDER"
    )


# ------------------------------------------------------------
# MODEL 1
# ------------------------------------------------------------

print(
    "\n[3] MODEL 1 — IMPERSONATION DETECTION"
)

print("-" * 90)

print(
    "Prediction       :",
    model1_prediction
)

print(
    "Legitimate       :",
    f"{legitimate_probability:.4f}"
)

print(
    "Impersonation    :",
    f"{impersonation_probability:.4f}"
)


# ------------------------------------------------------------
# MODEL 2
# ------------------------------------------------------------

print(
    "\n[4] MODEL 2 — ATTACK TYPE CLASSIFICATION"
)

print("-" * 90)

print(
    "Attack type      :",
    model2_prediction
)

print(
    "Confidence       :",
    f"{model2_confidence:.4f}"
)


print(
    "\nTop 3 attack types:"
)


for attack, probability in top3_attacks:

    print(
        f"  {attack:<25} "
        f"{probability:.4f}"
    )


# ------------------------------------------------------------
# MODEL 3
# ------------------------------------------------------------

print(
    "\n[5] MODEL 3 — BEHAVIORAL ANALYSIS"
)

print("-" * 90)

print(
    "Status           :",
    model3_status
)


if model3_score is not None:

    print(
        "Behavior score   :",
        f"{model3_score:.4f}"
    )

    print(
        "Time match       :",
        behavior_details["time_match"]
    )

    print(
        "Day match        :",
        behavior_details["day_match"]
    )

    print(
        "Domain match     :",
        behavior_details["domain_match"]
    )

    print(
        "Length match     :",
        behavior_details["length_match"]
    )

    print(
        "Length ratio     :",
        (
            f"{behavior_details['length_ratio']:.4f}"
            if behavior_details["length_ratio"]
            is not None
            else "N/A"
        )
    )

else:

    print(
        "Behavior score   : N/A"
    )


# ------------------------------------------------------------
# TECHNICAL / CONTENT
# ------------------------------------------------------------

print(
    "\n[6] TECHNICAL / CONTENT ANALYSIS"
)

print("-" * 90)

print(
    "Sender domain    :",
    sender_domain
)

print(
    "Receiver domain  :",
    receiver_domain
)

print(
    "Public email     :",
    public_email
)

print(
    "Domain mismatch  :",
    (
        identity_found
        and
        not sender_domain_matches_identity
    )
)

print(
    "Reply-To mismatch:",
    not reply_to_matches_sender
)

print(
    "Urgency          :",
    urgency_detected
)

print(
    "Sensitive request:",
    sensitive_request_detected
)

print(
    "Has link         :",
    has_link
)

print(
    "Link count       :",
    len(links)
)

print(
    "Has attachment   :",
    has_attachment
)


# ------------------------------------------------------------
# RISK ENGINE
# ------------------------------------------------------------

print(
    "\n[7] RISK / EVIDENCE FUSION"
)

print("-" * 90)


for name, value in risk_contributions.items():

    print(
        f"{name:<32}: +{value:.2f}"
    )


print("-" * 90)

print(
    "FINAL RISK SCORE :",
    f"{risk_score:.2f} / 100"
)

print(
    "RISK LEVEL       :",
    risk_band
)

print(
    "NOTE             : Engineering risk score, "
    "not a calibrated probability."
)


# ------------------------------------------------------------
# EXPLANATION
# ------------------------------------------------------------

print(
    "\n[8] EXPLANATION"
)

print("-" * 90)


for i, reason in enumerate(
    reasons,
    start=1
):

    print(
        f"{i}. {reason}"
    )


print("\n" + "=" * 90)

print(
    "                 CYBERGUARD ANALYSIS COMPLETE"
)

print("=" * 90)



                 CYBERGUARD — FINAL ANALYSIS

[1] EMAIL
------------------------------------------------------------------------------------------
Sender           : Rajesh Kumar
Sender email     : raj.kumar@gmail.com
Receiver         : Finance Officer
Receiver email   : finance@example.org
Subject          : Urgent: Verify your account immediately
Date/time        : 2026-09-23 14:30

[2] IDENTITY ANALYSIS — DATASET 1
------------------------------------------------------------------------------------------
Identity found   : False
Identity ambiguous: False
Identity ID      : None
Status           : UNKNOWN SENDER

[3] MODEL 1 — IMPERSONATION DETECTION
------------------------------------------------------------------------------------------
Prediction       : IMPERSONATION
Legitimate       : 0.3049
Impersonation    : 0.6951

[4] MODEL 2 — ATTACK TYPE CLASSIFICATION
------------------------------------------------------------------------------------------
Attack type      : CREDENTIA

In [21]:

import sys
import subprocess
print(sys.executable)

d:\abhijkk\my_env\Scripts\python.exe


In [22]:
import sys
import os

print("Jupyter Kernel:")
print("mlkernel (Python 3.13.2)")

print("\nPython executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nEnvironment prefix:")
print(sys.prefix)

print("\nBase prefix:")
print(sys.base_prefix)

print("\nIs this a virtual environment?")
print(sys.prefix != sys.base_prefix)

print("\nEnvironment folder exists?")
print(os.path.isdir(sys.prefix))

print("\nExpected environment:")
print(r"D:\abhijitkk\my_env")

print("\nCorrect environment?")
print(
    os.path.normcase(sys.prefix)
    == os.path.normcase(r"D:\abhijitkk\my_env")
)

Jupyter Kernel:
mlkernel (Python 3.13.2)

Python executable:
d:\abhijkk\my_env\Scripts\python.exe

Python version:
3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]

Environment prefix:
d:\abhijkk\my_env

Base prefix:
C:\Users\ASUS\AppData\Local\Programs\Python\Python313

Is this a virtual environment?
True

Environment folder exists?
True

Expected environment:
D:\abhijitkk\my_env

Correct environment?
False


In [23]:
import sys
import subprocess

print("Installing LLM dependencies into:")
print(sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "google-generativeai",
    "python-dotenv"
])

print("\nInstallation completed successfully.")

Installing LLM dependencies into:
d:\abhijkk\my_env\Scripts\python.exe

Installation completed successfully.
